# **Clasificación de Sistemas**

### **Objetivos de Aprendizaje**

* Comprender las principales categorías de sistemas en modelado y simulación.
* Diferenciar entre sistemas estáticos/dinámicos, deterministas/estocásticos, continuos/discretos/híbridos y abiertos/cerrados.
* Aplicar la clasificación a ejemplos de ingeniería de software y casos de la vida real.

### **¿Qué es un sistema?**

Un **sistema** es un conjunto de elementos que **interactúan** entre sí dentro de un **límite** (frontera) para cumplir un propósito. En modelado, solemos describirlo con:

* **Estado** `x` (lo que “recuerda” el sistema): colas en un servidor, temperatura de una CPU, número de bugs abiertos.
* **Entradas** `u` (lo que lo afecta desde fuera): tasa de llegada de peticiones, voltaje, señales de control.
* **Parámetros** `θ` (constantes del modelo): capacidad del servidor, coeficientes de desgaste.
* **Salidas** `y` (lo que observamos/medimos): latencia, consumo energético, disponibilidad.
* **Ruido / incertidumbre** `w, v` (si lo hay): variabilidad de llegadas, fallos, mediciones imperfectas.
* **Frontera** (qué está dentro y qué fuera): ¿incluyo la base de datos externa? ¿el balanceador global? Definirla evita “fugas” conceptuales.

Representaciones típicas:

* **Continuo (tiempo real):**

  $$
  \dot{x}(t)=f\big(x(t),\,u(t),\,\theta,\,t\big)+w(t),\quad y(t)=g\big(x(t),\,u(t),\,\theta,\,t\big)+v(t)
  $$
* **Discreto (por pasos/eventos):**

  $$
  x_{k+1}=F(x_k,\,u_k,\,\theta,\,\varepsilon_k),\quad y_k=G(x_k,\,u_k,\,\theta,\,\eta_k)
  $$

> **ejemplo:**
>
> **Microservicio**: requests → cola → servidor → respuesta (estado: longitud de cola; entrada: patrón de tráfico; salida: latencia).


### **Importancia de clasificar sistemas antes de modelar**

La **clasificación** orienta todas las decisiones del estudio de simulación:

1. **Elección del formalismo y del motor de simulación**
   (discreto vs continuo, determinista vs estocástico, híbrido, etc.).
2. **Supuestos y datos**: qué medir (tasa de llegadas, distribución de servicio, constantes físicas), con qué granularidad y horizonte temporal.
3. **Complejidad computacional**: una elección inadecuada puede ser demasiado lenta (p. ej., simular evento-a-evento cuando bastaría un modelo estático Monte Carlo).
4. **Validez/credibilidad**: un modelo “elegante” pero mal tipado (p. ej., continuo para fenómenos con bajas cuentas y eventos raros) genera conclusiones engañosas.
5. **Trazabilidad hacia la decisión**: la tipología define **métricas de desempeño** y **experimentos** relevantes (latencia, throughput, SLA, costo, consumo, huella de carbono).

**Riesgos comunes si no clasificas bien:**

* **Granularidad inadecuada** (paso de tiempo muy grueso o muy fino).
* **Distribuciones mal elegidas**.
* **Ignorar retroalimentaciones** (cambios que afectan la entrada).
* **Confundir estático con dinámico** (usar Monte Carlo para un problema claramente transitorio).

## **Estáticos vs. Dinámicos**

* **Estáticos:** la salida depende solo de parámetros/entradas “en un instante”. No hay trayectoria temporal.
  * Forma típica: $y = h(x,\theta)$.
  * Útiles para **comparar configuraciones** “snapshot” (capacidad, costo, confiabilidad puntual).


* **Dinámicos:** el **estado** cambia con el tiempo y genera trayectorias.
  * Forma continua: $\dot{x}(t)=f\big(x(t),u(t),\theta,t\big)$ y $y(t)=g(\cdot)$.
  * Forma discreta/eventos: $x_{k+1}=F(x_k,u_k,\theta,\varepsilon_k)$.

### **Preguntas que ayudan a clasificar**

* ¿Importa **cómo** se llega al resultado (historia)? → Dinámico.
* ¿Solo comparas alternativas **sin evolución**? → Estático.

### **Métricas típicas**

* Estáticos: confiabilidad, costo esperado, prob. de incumplir una tarea.
* Dinámicos: latencia temporal, backlog (longitud de cola), tiempo de estabilización.

### **Errores comunes**

* Usar modelo estático cuando hay **transitorios** relevantes.
* Ignorar **escala de tiempo** (minutos vs segundos cambia la dinámica observada).

### **Ejemplos:**

* **Estático:** **costo mensual** en la nube con precios/demanda inciertos.
* **Dinámico:** **tráfico** en una red de microservicios con ráfagas (bursts).

## **Deterministas vs. Estocásticos**

* **Deterministas:** mismas condiciones iniciales → mismo resultado.

  * Adecuado si la variación es despreciable o controlada.
* **Estocásticos:** hay **aleatoriedad** en entradas, parámetros o dinámica interna.

  * Fuentes: **llegadas**, **tiempos de servicio**, **fallos**, **mediciones**.

### **Implicaciones de modelado**

* Determinista → un **escenario** por condición.
* Estocástico → necesitas **réplicas** (N corridas), **intervalos de confianza**, y, si es posible, **reducción de varianza**.

### **Métricas**

* Reporta medias, **percentiles** (p50/p95/p99) y **IC** (p. ej. 95%).
* Sensibilidad: ¿qué parámetros dominan la variabilidad?

### **Errores comunes**

* Asumir **exponencial** por defecto en tiempos de servicio (cuando la cola tiene colas pesadas).
* Subestimar el número de **réplicas** → conclusiones frágiles.

### **Ejemplos:**

* **Determinista:** desempeño de un **algoritmo** (complejidad temporal) en entrada fija.
* **Estocástico:** **tiempos de espera** en una cola con llegadas variables.


## **Continuos, Discretos e Híbridos**

* **Continuos:** el estado evoluciona de modo suave con el tiempo (ODE/DAE/PDE).

  * Útil cuando el nivel de agregación es alto y hay “muchos” elementos (leyes de gran número).
* **Discretos:** los cambios ocurren en **eventos** (llegadas, inicios/fin de servicio, fallos).

  * **simulación de eventos discretos (DES)** y **redes de colas**.
* **Híbridos:** combinan ambos; p. ej., una dinámica térmica continua con eventos de scheduling.

### **Acoplamiento en híbridos**

1. DES detecta un evento (p. ej., nueva carga).
2. Actualiza parámetros de la ODE (p. ej., potencia disipada).
3. Integra ODE hasta el **siguiente evento** o por $\Delta t$.
4. Repite.

### **Errores comunes**

* Discretizar mal una dinámica continua (paso demasiado grande → inestabilidad numérica).
* Modelar como continuos fenómenos **raros** que exigen eventos discretos.

### **Ejemplos:**

* **Continuo:** propagación de **calor en CPU** (ecuaciones de difusión simplificadas).
* **Discreto:** **clientes en una fila** (llegadas/servicios/eventos).
* **Híbrido:** **vehículo autónomo**: dinámica mecánica (ODE) + decisiones/eventos (DES).

## **Abiertos vs. Cerrados**

* **Abiertos:** intercambian “flujo” con el entorno (entran y salen entidades).

  * En colas: **tasa de llegada exógena** $\lambda$.
* **Cerrados:** **población fija** de entidades circulando (no entran/salen, solo se mueven).

  * En colas: N “Trabajos” que rotan en un centro de servicio.

### **Implicaciones**

* **Abiertos:** útiles para dimensionar **capacidad** con demanda externa.
* **Cerrados:** útiles para **stress testing** con carga fija (p. ej., threads de un generador).

### **Ejemplos:**

* **Cerrado:** banco de pruebas con **N hilos** que ejercitan un API (población constante).
* **Abierto:** **plataforma en producción** con llegadas de usuarios desde Internet.

## Árbol de decisión

1. ¿La salida depende del tiempo?

* **No** → Estático → Monte Carlo estático / análisis directo.
* **Sí** → Dinámico 

2. ¿Hay aleatoriedad relevante?

* **No** → Determinista → ODE/DAE (continuo) o lógica discreta.
* **Sí** → Estocástico 

3. ¿Evoluciona por eventos o de forma suave?

* **Eventos** → DES / Colas / Markov.
* **Suave** → SDE/ODE con ruido.
* **Ambas** → Híbrido (DES + ODE).

4. ¿Intercambia entidades con el entorno?

* **Sí** → Abierto.
* **No** → Cerrado.

## **Comparación General**

| Tipo de sistema | Característica principal         | Ejemplo en software        |
| --------------- | -------------------------------- | -------------------------- |
| Estático        | Independiente del tiempo         | Análisis de código fuente  |
| Dinámico        | Evoluciona con el tiempo         | Simulación de red          |
| Determinista    | No hay aleatoriedad              | Algoritmo de búsqueda      |
| Estocástico     | Incluye azar                     | Procesos en colas          |
| Continuo        | Variables cambian de forma suave | Temperatura CPU            |
| Discreto        | Cambia en saltos/eventos         | Scheduler SO               |
| Híbrido         | Combina ambos                    | Carro autónomo             |
| Abierto         | Interacción con entorno          | API en la nube             |
| Cerrado         | Aislado                          | El universo |


### **Relación entre tipo de sistema y técnica de simulación adecuada**

| Clasificación                     | Señales/rasgos                                     | Técnica                                | Herramientas típicas                               |
| --------------------------------- | -------------------------------------------------- | ----------------------------------------------------- | -------------------------------------------------- |
| **Estático**        | Salidas dependen de parámetros, no de trayectorias | **Monte Carlo estático**, muestreo estratificado, LHS | `numpy.random`, `scipy.stats`                      |
| **Dinámico determinista**         | Física/control sin ruido, trayectorias suaves      | **ODE/DAE**, dinámica de sistemas                     | `scipy.integrate.solve_ivp` |
| **Dinámico estocástico continuo** | Ruido de proceso/medición, difusión                | **SDE**, filtros                                      | `sdeint`, , `scipy`, filtros de Kalman |
| **Eventos discretos**             | Colas, servidores, procesos por **eventos**        | **DES**                 | `SimPy`                          |
| **Markoviano**                    | Transiciones entre estados, memoria limitada       | **Cadenas de Markov**                | `numpy`, `pymdptoolbox`                            |
| **Híbrido**                       | Partes continuas + lógicas/eventos                 | **Co-simulación híbrida**, DES + ODE                  | `SimPy` + `scipy.integrate`                        |
| **Abierto**                       | Flujos desde/hacia entorno                         | **Redes de colas**, **simulación acoplada**                  | `SimPy`, `networkx`                                |
| **Cerrado**                       | Población fija de “clientes”                       | **Redes de colas cerradas**, modelos de producto-forma    | DES                                     |